# H2 · Ajuste de distribuciones de probabilidad

**Estadística I y II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Herramienta*

**Propósito de aprendizaje**

- Comparar la distribución observada de una variable con un modelo teórico (normal, lognormal, exponencial, gamma, Weibull, uniforme, triangular).
- Estimar parámetros por máxima verosimilitud y comparar modelos con gráficos (densidad, Q-Q) y medidas de ajuste.
- Usar el modelo elegido para calcular probabilidades de interés.

**Duración sugerida:** Uso libre  
**Datos:** cualquier variable numérica (por defecto `IC03_tiempos_reparacion.csv`)

## Cómo usar este notebook

Ejecute las celdas en orden. En la sección 1 elija el archivo y la variable. La sección 3 ajusta automáticamente varios modelos y los ordena por calidad de ajuste. La sección 4 permite probar un modelo **escrito a mano** (por ejemplo, el que el estudiante propuso a partir del histograma).

**Parametrizaciones usadas** (las mismas de clase, no las de `scipy`):

| Modelo | Cómo escribirlo | Parámetros |
|---|---|---|
| Normal | `normal(mu, sigma)` | media y desviación estándar |
| Lognormal | `lognormal(theta, omega)` | media y desviación de ln(X) |
| Exponencial | `exponencial(lam)` | tasa λ (media = 1/λ) |
| Gamma | `gamma(alfa, beta)` | forma α y escala β (media = αβ) |
| Weibull | `weibull(alfa, beta)` | forma α y escala β |
| Uniforme | `uniforme(a, b)` | mínimo y máximo |
| Triangular | `triangular(a, c, b)` | mínimo, moda y máximo |

# 1. Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
sns.set_theme(style="whitegrid")
AZUL, CORAL = "#00ADC6", "#FF6F49"

def leer_datos(nombre, hoja=None):
    rutas = [Path(nombre)] + [Path(p) / "datos" / c / nombre for p in [".", "..", "../.."] for c in ["clasicos", "casos_industriales/csv", "casos_industriales/xlsx", "talleres"]]
    ruta = next((r for r in rutas if r.exists()), None)
    if ruta is None:
        try:
            from google.colab import files
            print(f"No encontré '{nombre}'. Súbalo:")
            ruta = Path(next(iter(files.upload())))
        except ImportError:
            raise FileNotFoundError(f"No encontré '{nombre}'.")
    if ruta.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(ruta, sheet_name=hoja or 0)
    return pd.read_csv(ruta, sep=None, engine="python")

In [ ]:
#@markdown ### **Archivo y variable**
archivo = "IC03_tiempos_reparacion.csv"  #@param {type:"string"}
hoja_excel = ""  #@param {type:"string"}
variable = "tiempo_reparacion_min"  #@param {type:"string"}
#@markdown Filtro opcional, por ejemplo `tipo_equipo == "Prensa"`
condicion = ""  #@param {type:"string"}
#@markdown ---
df = leer_datos(archivo, hoja_excel or None)
if condicion.strip():
    df = df.query(condicion)
if variable not in df.columns:
    raise KeyError(f"'{variable}' no está en la base. Columnas: {list(df.columns)}")
x = pd.to_numeric(df[variable], errors="coerce").dropna().to_numpy()
print(f"{variable}: n = {len(x)} datos válidos")

# 2. Descripción de la variable

In [ ]:
from scipy.stats import skew, kurtosis
q = np.percentile(x, [0, 25, 50, 75, 100])
resumen = pd.DataFrame({"valor": [len(x), *q, x.mean(), x.std(ddof=1), x.std(ddof=1) / x.mean(), skew(x, bias=False), kurtosis(x, bias=False)]},
    index=["n", "mínimo", "Q1", "mediana", "Q3", "máximo", "media", "desv. estándar", "coef. de variación", "asimetría", "curtosis (exceso)"])
display(resumen.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x, stat="density", kde=True, color=AZUL, ax=ax[0]); ax[0].set_title("Histograma y densidad (KDE)")
sns.boxplot(x=x, color=AZUL, ax=ax[1]); ax[1].set_title("Caja y bigotes")
plt.show()

**Preguntas guía para el estudiante:** ¿la variable puede tomar valores negativos? ¿Es simétrica o sesgada? ¿Tiene un mínimo o un máximo natural? Estas respuestas descartan modelos antes de ajustar: por ejemplo, un tiempo de reparación no puede ser normal si la asimetría es fuerte, y una variable acotada en [a, b] sugiere uniforme o triangular.

# 3. Ajuste automático por máxima verosimilitud

In [ ]:
# Cada modelo: (nombre, función de ajuste que devuelve un objeto scipy "congelado", texto con parámetros)
def ajustar_modelos(x):
    res = []
    mu, s = stats.norm.fit(x)
    res.append(("Normal", stats.norm(mu, s), f"normal({mu:.4g}, {s:.4g})", 2))
    if x.min() > 0:
        sh, _, sc = stats.lognorm.fit(x, floc=0)
        res.append(("Lognormal", stats.lognorm(sh, 0, sc), f"lognormal({np.log(sc):.4g}, {sh:.4g})", 2))
        _, sc = stats.expon.fit(x, floc=0)
        res.append(("Exponencial", stats.expon(0, sc), f"exponencial({1/sc:.4g})", 1))
        a, _, sc = stats.gamma.fit(x, floc=0)
        res.append(("Gamma", stats.gamma(a, 0, sc), f"gamma({a:.4g}, {sc:.4g})", 2))
        c, _, sc = stats.weibull_min.fit(x, floc=0)
        res.append(("Weibull", stats.weibull_min(c, 0, sc), f"weibull({c:.4g}, {sc:.4g})", 2))
    a, b = x.min(), x.max()
    res.append(("Uniforme", stats.uniform(a, b - a), f"uniforme({a:.4g}, {b:.4g})", 2))
    c, loc, sc = stats.triang.fit(x)
    res.append(("Triangular", stats.triang(c, loc, sc), f"triangular({loc:.4g}, {loc + c*sc:.4g}, {loc + sc:.4g})", 3))
    return res

modelos = ajustar_modelos(x)
filas = []
for nombre, dist, texto, k in modelos:
    logL = np.sum(dist.logpdf(x))
    ks = stats.kstest(x, dist.cdf).statistic
    filas.append({"modelo": nombre, "parámetros": texto, "log-verosimilitud": logL,
                  "AIC": 2 * k - 2 * logL, "estadístico KS (D)": ks})
tabla_ajuste = pd.DataFrame(filas).sort_values("AIC").reset_index(drop=True)
display(tabla_ajuste.style.format({"log-verosimilitud": "{:.1f}", "AIC": "{:.1f}", "estadístico KS (D)": "{:.4f}"}))
print("Menor AIC y menor D indican mejor ajuste. El texto de 'parámetros' se puede copiar en la sección 4.")

> **Nota técnica.** El p-valor de Kolmogorov-Smirnov no es válido cuando los parámetros se estiman con los mismos datos (resulta demasiado optimista). Por eso aquí solo se reporta el estadístico D como medida de distancia. Para una prueba formal de normalidad con parámetros estimados use Shapiro-Wilk o Lilliefors (notebook H3).

In [ ]:
#@markdown ### **Comparar gráficamente los mejores modelos**
cuantos_modelos = 3  #@param {type:"integer"}
#@markdown ---
mejores = [m for m in modelos if m[0] in tabla_ajuste["modelo"].head(cuantos_modelos).tolist()]
grid = np.linspace(x.min(), x.max(), 400)
colores = [CORAL, "#25A687", "#7B6FD0", "#F2B134", "#333333"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(x, bins="auto", density=True, color=AZUL, alpha=.45, label="datos")
for (nombre, dist, texto, _), col in zip(mejores, colores):
    ax[0].plot(grid, dist.pdf(grid), color=col, lw=2, label=texto)
ax[0].set_title("Histograma vs. densidades ajustadas"); ax[0].legend(fontsize=8)
xs = np.sort(x); pp = (np.arange(1, len(xs) + 1) - .5) / len(xs)
ax[1].step(xs, pp, where="post", color=AZUL, label="empírica")
for (nombre, dist, texto, _), col in zip(mejores, colores):
    ax[1].plot(grid, dist.cdf(grid), color=col, lw=1.8, label=nombre)
ax[1].set_title("Función de distribución acumulada"); ax[1].legend(fontsize=8)
plt.show()

# 4. Probar un modelo propuesto

In [ ]:
# Constructores con la parametrización de clase
normal      = lambda mu, sigma: stats.norm(mu, sigma)
lognormal   = lambda theta, omega: stats.lognorm(omega, 0, np.exp(theta))
exponencial = lambda lam: stats.expon(0, 1 / lam)
gamma       = lambda alfa, beta: stats.gamma(alfa, 0, beta)
weibull     = lambda alfa, beta: stats.weibull_min(alfa, 0, beta)
uniforme    = lambda a, b: stats.uniform(a, b - a)
triangular  = lambda a, c, b: stats.triang((c - a) / (b - a), a, b - a)
CONSTRUCTORES = dict(normal=normal, lognormal=lognormal, exponencial=exponencial, gamma=gamma,
                     weibull=weibull, uniforme=uniforme, triangular=triangular)

def interpretar(texto):
    """Convierte un texto como 'gamma(2, 30)' en una distribución, sin usar eval sobre código arbitrario."""
    import re
    m = re.fullmatch(r"\s*(\w+)\s*\(([^)]*)\)\s*", texto)
    if not m or m.group(1) not in CONSTRUCTORES:
        raise ValueError(f"Escriba algo como normal(10, 2). Modelos disponibles: {list(CONSTRUCTORES)}")
    args = [float(v) for v in m.group(2).split(",")]
    return CONSTRUCTORES[m.group(1)](*args)

In [ ]:
#@markdown ### **Modelo propuesto**
modelo_propuesto = "gamma(2, 50)"  #@param {type:"string"}
#@markdown ---
dist = interpretar(modelo_propuesto)
p = np.arange(.05, 1, .05)
tabla_q = pd.DataFrame({"probabilidad": p, "cuantil empírico": np.quantile(x, p), "cuantil teórico": dist.ppf(p)})
tabla_q["diferencia"] = tabla_q["cuantil empírico"] - tabla_q["cuantil teórico"]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
grid = np.linspace(x.min(), x.max(), 400)
sns.kdeplot(x, fill=True, color=AZUL, ax=ax[0], label="KDE de los datos")
ax[0].plot(grid, dist.pdf(grid), "--", color=CORAL, lw=2, label=modelo_propuesto)
ax[0].legend(); ax[0].set_title("Densidad observada vs. propuesta")
teo = dist.ppf(pp)
ax[1].scatter(teo, xs, s=10, color=AZUL)
lim = [min(teo.min(), xs.min()), max(teo.max(), xs.max())]
ax[1].plot(lim, lim, "--", color=CORAL); ax[1].set_xlabel("cuantiles teóricos"); ax[1].set_ylabel("cuantiles observados")
ax[1].set_title("Gráfico Q-Q")
plt.show()
display(tabla_q.round(3))
print(f"Log-verosimilitud = {np.sum(dist.logpdf(x)):.1f}   |   D de Kolmogorov-Smirnov = {stats.kstest(x, dist.cdf).statistic:.4f}")

# 5. Calcular probabilidades con el modelo

In [ ]:
#@markdown ### **Probabilidad de un intervalo**
#@markdown Deje vacío un límite para un intervalo abierto. Ejemplo: P(X > 120) → inferior = 120, superior vacío.
limite_inferior = "120"  #@param {type:"string"}
limite_superior = ""  #@param {type:"string"}
#@markdown ---
li = float(limite_inferior) if limite_inferior.strip() else -np.inf
ls = float(limite_superior) if limite_superior.strip() else np.inf
p_modelo = dist.cdf(ls) - dist.cdf(li)
p_datos = np.mean((x > li) & (x <= ls))
print(f"P({li} < X ≤ {ls})  según el modelo {modelo_propuesto}: {p_modelo:.4f}")
print(f"Proporción observada en los datos:                {p_datos:.4f}")